
# Week 2 – Deep Learning with PyTorch

**AI/ML Internship – Skill Set Go EduTech**

This notebook covers all Week 2 requirements:

- **2.1 Neural Network in PyTorch:** tensors, datasets, dataloaders, loss function, training and evaluation.
- **2.2 CNN Image Classifier:** MNIST image preparation, CNN architecture, training and accuracy evaluation.
- **2.3 Hyperparameter Experimentation:** controlled experiments with optimizers, learning rates, batch sizes and epochs.
- **2.4 Deep Learning Technical Report:** comparison, findings, limitations and future improvements.

**Dataset:** MNIST handwritten digits  
**Framework:** PyTorch  
**Environment:** Google Colab



## 0. Setup

The notebook is designed for Google Colab. GPU is recommended but the code also works on CPU.

In Colab, you can enable a GPU from:

**Runtime → Change runtime type → T4 GPU**

The code automatically detects whether CUDA is available.


In [ ]:

import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split

from torchvision import datasets, transforms

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch version:", torch.__version__)
print("Device:", device)


## 1. Load and Inspect the MNIST Dataset

In [ ]:

DATA_DIR = "./data"

transform = transforms.ToTensor()

train_full = datasets.MNIST(
    root=DATA_DIR,
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.MNIST(
    root=DATA_DIR,
    train=False,
    download=True,
    transform=transform
)

train_size = int(0.9 * len(train_full))
val_size = len(train_full) - train_size

train_dataset, val_dataset = random_split(
    train_full,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(SEED)
)

print("Training samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Test samples:", len(test_dataset))
print("Image shape:", train_full[0][0].shape)
print("Number of classes:", len(train_full.classes))
print("Classes:", train_full.classes)


In [ ]:

# Display sample images
fig, axes = plt.subplots(2, 5, figsize=(10, 4))

for ax, (image, label) in zip(axes.ravel(), [train_full[i] for i in range(10)]):
    ax.imshow(image.squeeze(), cmap="gray")
    ax.set_title(f"Label: {label}")
    ax.axis("off")

plt.tight_layout()
plt.show()



# 2.1 Neural Network in PyTorch

A feed-forward neural network will be trained on flattened MNIST images.

The model contains:
- Input layer: 28 × 28 = 784 features
- Hidden layer 1
- ReLU activation
- Hidden layer 2
- ReLU activation
- Output layer with 10 classes

Cross-entropy loss is used for multi-class classification.


In [ ]:

class FeedForwardNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.network = nn.Sequential(
            nn.Linear(28 * 28, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 10)
        )

    def forward(self, x):
        x = self.flatten(x)
        return self.network(x)


def make_loaders(batch_size=64):
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    return train_loader, val_loader, test_loader


def train_model(model, train_loader, val_loader, criterion, optimizer, epochs=3):
    history = {
        "train_loss": [],
        "val_loss": [],
        "train_accuracy": [],
        "val_accuracy": []
    }

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        train_correct = 0
        train_total = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)
            predictions = outputs.argmax(dim=1)
            train_correct += (predictions == labels).sum().item()
            train_total += labels.size(0)

        train_loss = running_loss / train_total
        train_acc = train_correct / train_total

        model.eval()
        val_loss_total = 0.0
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)

                val_loss_total += loss.item() * images.size(0)
                predictions = outputs.argmax(dim=1)
                val_correct += (predictions == labels).sum().item()
                val_total += labels.size(0)

        val_loss = val_loss_total / val_total
        val_acc = val_correct / val_total

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_accuracy"].append(train_acc)
        history["val_accuracy"].append(val_acc)

        print(
            f"Epoch {epoch+1}/{epochs} | "
            f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
            f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}"
        )

    return history


def evaluate_model(model, data_loader):
    model.eval()
    all_labels = []
    all_predictions = []

    with torch.no_grad():
        for images, labels in data_loader:
            images = images.to(device)
            outputs = model(images)
            predictions = outputs.argmax(dim=1).cpu().numpy()

            all_predictions.extend(predictions)
            all_labels.extend(labels.numpy())

    accuracy = accuracy_score(all_labels, all_predictions)

    return np.array(all_labels), np.array(all_predictions), accuracy


In [ ]:

BATCH_SIZE = 64
EPOCHS = 3

train_loader, val_loader, test_loader = make_loaders(BATCH_SIZE)

ffn = FeedForwardNN().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(ffn.parameters(), lr=0.001)

ffn_history = train_model(
    ffn,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    epochs=EPOCHS
)

ffn_labels, ffn_predictions, ffn_test_accuracy = evaluate_model(ffn, test_loader)

print(f"Feed-forward neural network test accuracy: {ffn_test_accuracy:.4f}")


In [ ]:

# Plot training history
epochs_range = range(1, EPOCHS + 1)

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(epochs_range, ffn_history["train_loss"], marker="o", label="Train Loss")
plt.plot(epochs_range, ffn_history["val_loss"], marker="o", label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Feed-Forward Network Loss")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs_range, ffn_history["train_accuracy"], marker="o", label="Train Accuracy")
plt.plot(epochs_range, ffn_history["val_accuracy"], marker="o", label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Feed-Forward Network Accuracy")
plt.legend()

plt.tight_layout()
plt.show()



## 2.2 CNN Image Classifier

A Convolutional Neural Network is better suited to image data because convolutional layers learn local visual patterns.

Architecture:
- Convolution → ReLU → Max Pooling
- Convolution → ReLU → Max Pooling
- Flatten
- Fully connected layer
- Output layer with 10 classes


In [ ]:

class CNNClassifier(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)


In [ ]:

cnn = CNNClassifier().to(device)

cnn_criterion = nn.CrossEntropyLoss()
cnn_optimizer = optim.Adam(cnn.parameters(), lr=0.001)

cnn_history = train_model(
    cnn,
    train_loader,
    val_loader,
    cnn_criterion,
    cnn_optimizer,
    epochs=3
)

cnn_labels, cnn_predictions, cnn_test_accuracy = evaluate_model(cnn, test_loader)

print(f"CNN test accuracy: {cnn_test_accuracy:.4f}")

print("\nClassification Report:")
print(
    classification_report(
        cnn_labels,
        cnn_predictions,
        target_names=[str(i) for i in range(10)]
    )
)


In [ ]:

# CNN learning curves
epochs_range = range(1, len(cnn_history["train_loss"]) + 1)

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(epochs_range, cnn_history["train_loss"], marker="o", label="Train Loss")
plt.plot(epochs_range, cnn_history["val_loss"], marker="o", label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("CNN Loss")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs_range, cnn_history["train_accuracy"], marker="o", label="Train Accuracy")
plt.plot(epochs_range, cnn_history["val_accuracy"], marker="o", label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("CNN Accuracy")
plt.legend()

plt.tight_layout()
plt.show()


In [ ]:

# Confusion matrix for the CNN
cm = confusion_matrix(cnn_labels, cnn_predictions)

plt.figure(figsize=(8, 6))
plt.imshow(cm, cmap="Blues")
plt.title("CNN Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.colorbar()

plt.xticks(range(10))
plt.yticks(range(10))

for i in range(10):
    for j in range(10):
        plt.text(j, i, cm[i, j], ha="center", va="center")

plt.tight_layout()
plt.show()


In [ ]:

# Show a few predictions
sample_images, sample_labels = next(iter(test_loader))
sample_images = sample_images.to(device)

with torch.no_grad():
    sample_outputs = cnn(sample_images)
    sample_predictions = sample_outputs.argmax(dim=1).cpu()

plt.figure(figsize=(12, 5))

for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(sample_images[i].cpu().squeeze(), cmap="gray")
    plt.title(f"True: {sample_labels[i]} | Pred: {sample_predictions[i]}")
    plt.axis("off")

plt.tight_layout()
plt.show()



# 2.3 Hyperparameter Experimentation

The purpose of this section is to run controlled experiments rather than changing many settings at once.

We will compare:
- Optimizer: Adam vs SGD
- Learning rate: 0.001 vs 0.01
- Batch size: 64 vs 128
- Epochs: kept consistent at 2 for the experiment stage

The same train/validation split and model architecture are used so the comparison is meaningful.

For the final notebook, the main CNN above is the trained CNN deliverable. These experiments provide the required experiment log and comparison evidence.


In [ ]:

def run_cnn_experiment(optimizer_name, learning_rate, batch_size, epochs=2):
    exp_train_loader, exp_val_loader, _ = make_loaders(batch_size)

    model = CNNClassifier().to(device)
    criterion = nn.CrossEntropyLoss()

    if optimizer_name == "Adam":
        optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    elif optimizer_name == "SGD":
        optimizer = optim.SGD(model.parameters(), lr=learning_rate, momentum=0.9)
    else:
        raise ValueError("Unsupported optimizer")

    history = train_model(
        model,
        exp_train_loader,
        exp_val_loader,
        criterion,
        optimizer,
        epochs=epochs
    )

    best_val_accuracy = max(history["val_accuracy"])
    final_val_accuracy = history["val_accuracy"][-1]

    return {
        "optimizer": optimizer_name,
        "learning_rate": learning_rate,
        "batch_size": batch_size,
        "epochs": epochs,
        "best_validation_accuracy": best_val_accuracy,
        "final_validation_accuracy": final_val_accuracy
    }


experiment_configs = [
    {"optimizer_name": "Adam", "learning_rate": 0.001, "batch_size": 64, "epochs": 2},
    {"optimizer_name": "Adam", "learning_rate": 0.01,  "batch_size": 64, "epochs": 2},
    {"optimizer_name": "SGD",  "learning_rate": 0.001, "batch_size": 64, "epochs": 2},
    {"optimizer_name": "Adam", "learning_rate": 0.001, "batch_size": 128, "epochs": 2},
]

experiment_results = []

for i, config in enumerate(experiment_configs, start=1):
    print(f"\n===== Experiment {i} =====")
    print(config)

    result = run_cnn_experiment(**config)
    result["experiment"] = i
    experiment_results.append(result)

experiment_df = pd.DataFrame(experiment_results)

experiment_df = experiment_df[
    [
        "experiment",
        "optimizer",
        "learning_rate",
        "batch_size",
        "epochs",
        "best_validation_accuracy",
        "final_validation_accuracy"
    ]
]

experiment_df


In [ ]:

# Visual comparison of the experiments
plt.figure(figsize=(9, 5))

labels = [
    f"E{i}: {opt}, lr={lr}, bs={bs}"
    for i, opt, lr, bs in zip(
        experiment_df["experiment"],
        experiment_df["optimizer"],
        experiment_df["learning_rate"],
        experiment_df["batch_size"]
    )
]

plt.bar(labels, experiment_df["best_validation_accuracy"])
plt.ylabel("Best Validation Accuracy")
plt.title("Hyperparameter Experiment Comparison")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

best_experiment = experiment_df.loc[
    experiment_df["best_validation_accuracy"].idxmax()
]

print("Best experiment by validation accuracy:")
print(best_experiment)


In [ ]:

# Save the experiment log
experiment_df.to_csv("week2_experiment_log.csv", index=False)

print("Saved: week2_experiment_log.csv")



# 2.4 Deep Learning Technical Report

## Objective
The objective of Week 2 was to develop practical deep-learning skills using PyTorch. The work covered tensors and data loading, a feed-forward neural network, convolutional neural networks, controlled hyperparameter experiments, and model evaluation.

## Dataset
The MNIST handwritten-digit dataset was used. It contains grayscale images of handwritten digits from 0 to 9. The images have a resolution of 28 × 28 pixels.

## Feed-Forward Neural Network
The first model flattened each 28 × 28 image into 784 input features and passed them through two fully connected hidden layers with ReLU activation. Cross-entropy loss was used because the task is multi-class classification.

The model was trained using the Adam optimizer and evaluated on the held-out test set.

## CNN Image Classifier
The second model used two convolutional blocks with ReLU activation and max pooling, followed by fully connected layers. This architecture allows the network to learn spatial patterns from the images.

The CNN was evaluated using test accuracy, a classification report, and a confusion matrix.

## Hyperparameter Experiments
Controlled experiments were performed by changing the optimizer, learning rate, and batch size while keeping the model architecture and data split consistent. The experiment log records the configuration and validation accuracy for each run.

The best configuration should be identified from the generated `experiment_df` table rather than being assumed in advance. This makes the conclusion data-driven.

## Results
The exact accuracy values are generated when the notebook is executed because training is performed in the Colab runtime. The notebook records:
- Feed-forward network test accuracy
- CNN test accuracy
- CNN classification metrics
- CNN confusion matrix
- Hyperparameter experiment comparison

## Key Learning Outcomes
This work provided practical experience with:
1. PyTorch tensors and datasets.
2. DataLoader-based mini-batch training.
3. Neural-network architecture design.
4. Loss functions and backpropagation.
5. CNN-based image classification.
6. Hyperparameter experimentation.
7. Model evaluation and interpretation.

## Limitations
The experiments use a relatively small number of epochs to keep the notebook practical for a student Google Colab environment. Results can vary slightly between runs despite using a fixed random seed, especially when GPU operations are involved.

## Future Improvements
Possible improvements include longer training, learning-rate scheduling, data augmentation, additional CNN architectures, systematic hyperparameter search, and evaluation on additional image datasets.

## Conclusion
Week 2 strengthened the transition from traditional machine learning to deep learning. The completed notebook demonstrates the full workflow from dataset preparation and model construction to training, evaluation, experimentation, and technical documentation.


In [ ]:

# Save trained models and final report-related outputs
torch.save(ffn.state_dict(), "week2_feedforward_network.pth")
torch.save(cnn.state_dict(), "week2_cnn_mnist.pth")

results_summary = pd.DataFrame({
    "Model": ["Feed-Forward Neural Network", "CNN Image Classifier"],
    "Test Accuracy": [ffn_test_accuracy, cnn_test_accuracy]
})

results_summary.to_csv("week2_model_results.csv", index=False)

print("Saved files:")
print("- week2_feedforward_network.pth")
print("- week2_cnn_mnist.pth")
print("- week2_model_results.csv")
print("- week2_experiment_log.csv")
results_summary



# Final Submission Checklist

Before submitting Week 2:

- [ ] 2.1 feed-forward PyTorch network completed
- [ ] 2.2 CNN image classifier completed
- [ ] CNN trained model file saved
- [ ] Accuracy report / classification report generated
- [ ] 2.3 experiments completed with comparison table
- [ ] Experiment log CSV saved
- [ ] 2.4 technical report completed in this notebook
- [ ] Notebook run from top to bottom without errors
- [ ] GitHub repository updated
- [ ] Screenshots/evidence captured
- [ ] All links tested in an incognito/private window
- [ ] Week 2 Google Form submitted
